# Vid2Log — Auto-Discover Training Classes from a Demo Video

Given a raw 2-3 minute demo recording and **no trained model yet**, this notebook automatically:

1. Samples frames from the video,
2. Embeds every sampled frame with a vision transformer (**DINOv2**) to capture what's visually on screen,
3. Clusters those embeddings (**HDBSCAN**) to discover the video's *unique recurring screens/actions* — with no need to know in advance how many classes exist,
4. Proposes a short name for each cluster using an image-captioning model,
5. Gives you an **interactive review UI** to rename any class and merge clusters that are really the same screen,
6. Exports a diversity-sampled set of images per surviving class into folders shaped exactly like vid2log's own **Train a model** tab expects (one folder per class = one `ClassDraft`).

**Pipeline:** `Video -> Frame Sampling -> DINOv2 Embeddings -> HDBSCAN Clustering -> Auto-Naming -> Rename/Merge Review -> Diversity Subsampling -> Export`

This is deliberately scoped to *class discovery*, not full step-by-step event detection (compare with `Vid2Log_UniqueFrameExtraction.ipynb`, which detects individual navigation/scroll/dialog *events* along a timeline) — here every sampled frame gets assigned to a class, since the goal is a labeled image dataset per recurring screen, not a list of transition moments.

## 0. Setup — install dependencies

In [ ]:
!pip install -q transformers timm opencv-python-headless scikit-learn hdbscan ipywidgets tqdm pandas pillow matplotlib

In [ ]:
import os
import shutil
import json

import numpy as np
import pandas as pd
import cv2
import torch
from PIL import Image
from tqdm.auto import tqdm
from sklearn.preprocessing import normalize
import hdbscan
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 1. Point to your video

Upload your demo recording to this Colab session (drag it into the Files pane on the left, or use the commented snippet below), then set `VIDEO_PATH`.

In [ ]:
VIDEO_PATH = "/content/sample_video.mp4"  # <-- change this to your uploaded video

# Optional: upload from your local machine in Colab
# from google.colab import files
# uploaded = files.upload()
# VIDEO_PATH = list(uploaded.keys())[0]

## Step 1 — Frame Sampling

**What we're doing:** reading the video with OpenCV and keeping frames at a fixed rate (default 2 fps, matching vid2log's own backend default in `video_pipeline.py`) instead of every raw frame — consecutive frames in a screen recording are almost always near-identical, so this keeps the amount of downstream work proportional to how much actually *happens* in the video, not its raw frame rate.

**Technique/library:** OpenCV (`cv2`) frame-by-frame video decoding.

In [ ]:
def sample_frames(video_path, desired_fps=2):
    video = cv2.VideoCapture(video_path)
    if not video.isOpened():
        raise IOError(f"Could not open video: {video_path}")

    src_fps = video.get(cv2.CAP_PROP_FPS) or 30.0
    interval = max(1, int(round(src_fps / desired_fps)))

    frames, timestamps = [], []
    count = 0
    while True:
        ret, frame = video.read()
        if not ret:
            break
        if count % interval == 0:
            frames.append(frame)  # BGR, as read by OpenCV
            timestamps.append(count / src_fps)
        count += 1

    video.release()
    return frames, timestamps


frames, timestamps = sample_frames(VIDEO_PATH, desired_fps=2)
print(f"Sampled {len(frames)} frames from the video (~{timestamps[-1]:.1f}s at 2 fps).")

## Step 2 — Visual Embeddings (DINOv2)

**What we're doing:** converting *every* sampled frame into a single numeric vector that summarizes its visual content, so "how similar are two screens" becomes "how close are two vectors" instead of slow/noisy raw pixel comparison.

**Model/technique:** **DINOv2 (`facebook/dinov2-base`)** — a self-supervised Vision Transformer from Meta AI. Chosen over CLIP/SigLIP specifically for this step because benchmarks show it produces tighter, more separated clusters on pure visual-similarity tasks (no text pairing needed, since we're not trying to match images to captions here — we're trying to tell "which screens look alike"). The CLS token from the last hidden layer is taken as each frame's 768-dimensional embedding.

In [ ]:
from transformers import AutoImageProcessor, AutoModel

DINO_MODEL_ID = "facebook/dinov2-base"
processor = AutoImageProcessor.from_pretrained(DINO_MODEL_ID)
dino_model = AutoModel.from_pretrained(DINO_MODEL_ID).to(device)
dino_model.eval()


@torch.no_grad()
def get_embedding(frame_bgr):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(frame_rgb)
    inputs = processor(images=pil_img, return_tensors="pt").to(device)
    outputs = dino_model(**inputs)
    # CLS token: a single 768-d vector summarizing the whole frame.
    return outputs.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()


embeddings = np.stack([get_embedding(f) for f in tqdm(frames, desc="Embedding frames")])
print(f"Computed {embeddings.shape[0]} embeddings of dimension {embeddings.shape[1]}.")

## Step 3 — Unsupervised Clustering (HDBSCAN)

**What we're doing:** grouping frames whose embeddings are close together into clusters — each cluster is a candidate "unique screen/action" in the demo, discovered with *no prior knowledge of how many classes exist*.

**Technique:** **HDBSCAN**, density-based clustering. Chosen over k-means specifically because you don't know the true number of screens ahead of time (k-means requires guessing that), and HDBSCAN naturally treats ambiguous/transitional frames as noise rather than forcing every frame into a cluster.

Embeddings are L2-normalized first so plain Euclidean distance behaves as a stand-in for cosine similarity — this sidesteps HDBSCAN's patchier support for a raw `cosine` metric while still clustering on visual *similarity*, not raw vector magnitude.

`min_cluster_size` is the main knob: it's how many sampled frames must share a "look" before it counts as a real recurring screen rather than a one-off blip. At 2 fps, `min_cluster_size=5` means "stayed on screen for roughly 2.5 real seconds" — raise it for longer/slower demos, lower it if you expect brief-but-real screens to be getting dropped.

In [ ]:
normed_embeddings = normalize(embeddings, norm="l2")

MIN_CLUSTER_SIZE = 5  # tune per your video's pacing — see markdown above

clusterer = hdbscan.HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean")
raw_labels = clusterer.fit_predict(normed_embeddings)

n_noise = int((raw_labels == -1).sum())
n_clusters = len(set(raw_labels)) - (1 if n_noise else 0)
print(f"HDBSCAN found {n_clusters} candidate screen clusters ({n_noise} frames initially unassigned).")

## Step 4 — Reassign Unclustered Frames

**What we're doing:** HDBSCAN deliberately leaves ambiguous frames unlabeled (`-1`) rather than forcing them into a cluster it isn't confident about. For *this* use case, every sampled frame is a real moment from the user's demo and should end up in some class — so each noise frame gets folded into whichever real cluster's centroid it's closest to, instead of being discarded.

In [ ]:
def reassign_noise(embeddings, labels):
    labels = labels.copy()
    real_clusters = sorted(set(labels) - {-1})
    if not real_clusters:
        return labels
    centroids = np.stack([embeddings[labels == c].mean(axis=0) for c in real_clusters])
    noise_idx = np.where(labels == -1)[0]
    for i in noise_idx:
        dists = np.linalg.norm(centroids - embeddings[i], axis=1)
        labels[i] = real_clusters[int(np.argmin(dists))]
    return labels


final_labels = reassign_noise(normed_embeddings, raw_labels)
print("All frames now assigned to a class:", (final_labels != -1).all())

## Step 5 — Cluster Summaries & Representative Frames

**What we're doing:** for each cluster, finding its **medoid** — the member frame closest to the cluster's centroid — to use as the representative thumbnail for naming and review. This is more robust than just picking the first or a random frame, since it's the frame most "typical" of that class.

In [ ]:
def build_cluster_summaries(embeddings, labels):
    clusters = {}
    for cluster_id in sorted(set(labels)):
        member_idx = np.where(labels == cluster_id)[0]
        centroid = embeddings[member_idx].mean(axis=0)
        dists = np.linalg.norm(embeddings[member_idx] - centroid, axis=1)
        medoid_idx = int(member_idx[np.argmin(dists)])
        clusters[int(cluster_id)] = {
            "id": int(cluster_id),
            "name": f"Class {cluster_id + 1}",  # placeholder — replaced in Step 6
            "member_indices": member_idx.tolist(),
            "medoid_index": medoid_idx,
        }
    return clusters


clusters = build_cluster_summaries(normed_embeddings, final_labels)
print(f"{len(clusters)} clusters — sizes:",
      {cid: len(c["member_indices"]) for cid, c in clusters.items()})

## Step 6 — Auto-Name Each Cluster

**What we're doing:** captioning each cluster's representative frame and turning that into a short, class-name-like label.

**Model/technique:** **BLIP** (`Salesforce/blip-image-captioning-base`) via BLIP's own `BlipProcessor`/`BlipForConditionalGeneration` classes directly (not the generic `pipeline()` wrapper, whose task-name aliases have drifted across `transformers` versions). BLIP is used here specifically for reliability — it's small, CPU-friendly, and has a long-stable API, which matters for a notebook meant to just work on a fresh Colab runtime. Moondream2, Florence-2, or Gemma 3 are viable drop-in swaps for richer names if you have more GPU headroom (see the closing notes), and a commented-out Gemini API option is included below for a hosted-model upgrade.

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

BLIP_MODEL_ID = "Salesforce/blip-image-captioning-base"
blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL_ID)
blip_model = BlipForConditionalGeneration.from_pretrained(BLIP_MODEL_ID).to(device)
blip_model.eval()


@torch.no_grad()
def propose_name(frame_bgr, max_words=6):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(frame_rgb)
    inputs = blip_processor(images=pil_img, return_tensors="pt").to(device)
    output_ids = blip_model.generate(**inputs, max_new_tokens=20)
    caption = blip_processor.decode(output_ids[0], skip_special_tokens=True).strip()
    words = caption.split()[:max_words]
    return " ".join(words).title()


for cluster in tqdm(clusters.values(), desc="Naming clusters"):
    medoid_frame = frames[cluster["medoid_index"]]
    cluster["name"] = propose_name(medoid_frame)

print("Proposed class names:")
for c in clusters.values():
    print(f"  Class {c['id']}: \"{c['name']}\"  ({len(c['member_indices'])} frames)")

In [ ]:
# --- Optional: use a hosted vision model (Gemini) for nicer names instead ---
# Uncomment and set an API key to try this — costs a fraction of a cent per
# image via the Batch API, and can give noticeably more specific names than
# BLIP (e.g. "Checkout Confirmation" instead of "a screenshot of a form").
#
# import google.generativeai as genai
# genai.configure(api_key="YOUR_API_KEY")
# gemini = genai.GenerativeModel("gemini-2.5-flash")
#
# def propose_name_gemini(frame_bgr):
#     frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
#     pil_img = Image.fromarray(frame_rgb)
#     prompt = ("This is a screenshot from a screen-recorded app demo. "
#               "Name the screen/state/action shown in 2-4 words, title case, "
#               "no punctuation.")
#     response = gemini.generate_content([prompt, pil_img])
#     return response.text.strip()
#
# for cluster in clusters.values():
#     cluster["name"] = propose_name_gemini(frames[cluster["medoid_index"]])

## Step 7 — Preview Clusters

A quick visual sanity check before the interactive review: one thumbnail per cluster, with its proposed name and frame count.

In [ ]:
def preview_clusters(clusters, frames):
    n = len(clusters)
    cols = min(4, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, cluster in zip(axes, clusters.values()):
        frame_rgb = cv2.cvtColor(frames[cluster["medoid_index"]], cv2.COLOR_BGR2RGB)
        ax.imshow(frame_rgb)
        ax.set_title(f"{cluster['name']}\n({len(cluster['member_indices'])} frames)", fontsize=10)
        ax.axis("off")
    for ax in axes[len(clusters):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


preview_clusters(clusters, frames)

## Step 8 — Interactive Review: Rename & Merge

This is the human-in-the-loop step: tick 2+ checkboxes and click **"Merge selected"** to combine clusters that are really the same screen (their frames get pooled under the first selected class); edit any name directly in its text box, then click **"Save names"** once you're happy. Re-run this cell if you want to reset the review from the clusters computed above.

In [ ]:
review_state = {"clusters": {cid: dict(c, member_indices=list(c["member_indices"])) for cid, c in clusters.items()}}
row_widgets = {}
output_area = widgets.Output()


def frame_to_widget_image(frame_bgr, max_size=200):
    h, w = frame_bgr.shape[:2]
    scale = max_size / max(h, w)
    small = cv2.resize(frame_bgr, (max(1, int(w * scale)), max(1, int(h * scale))))
    ok, buf = cv2.imencode(".png", small)
    return widgets.Image(value=buf.tobytes(), format="png")


def sync_names_from_widgets():
    for cid, w in row_widgets.items():
        if cid in review_state["clusters"]:
            review_state["clusters"][cid]["name"] = w["name_box"].value


def on_merge_clicked(_):
    sync_names_from_widgets()
    selected = [cid for cid, w in row_widgets.items() if w["checkbox"].value]
    if len(selected) < 2:
        status_label.value = "Select at least 2 classes (checkboxes) to merge."
        return
    target_id = selected[0]
    target = review_state["clusters"][target_id]
    for cid in selected[1:]:
        other = review_state["clusters"].pop(cid)
        target["member_indices"].extend(other["member_indices"])
    status_label.value = f"Merged {len(selected)} classes into \"{target['name']}\"."
    refresh()


def on_save_clicked(_):
    sync_names_from_widgets()
    status_label.value = f"Saved {len(review_state['clusters'])} class names — run the export cells below."


merge_button = widgets.Button(description="Merge selected", button_style="warning")
save_button = widgets.Button(description="Save names", button_style="success")
status_label = widgets.Label("")
merge_button.on_click(on_merge_clicked)
save_button.on_click(on_save_clicked)


def refresh():
    row_widgets.clear()
    rows = []
    for cid, cluster in review_state["clusters"].items():
        checkbox = widgets.Checkbox(value=False, indent=False, layout=widgets.Layout(width="28px"))
        thumb = frame_to_widget_image(frames[cluster["medoid_index"]])
        name_box = widgets.Text(value=cluster["name"], layout=widgets.Layout(width="220px"))
        count_label = widgets.Label(f"{len(cluster['member_indices'])} frames")
        row_widgets[cid] = {"checkbox": checkbox, "name_box": name_box}
        rows.append(widgets.HBox([checkbox, thumb, name_box, count_label]))
    with output_area:
        clear_output(wait=True)
        display(widgets.VBox(rows + [widgets.HBox([merge_button, save_button]), status_label]))


display(widgets.HTML(
    "<b>Tick 2+ boxes and click \"Merge selected\" to combine classes that are really the same screen. "
    "Edit any name directly, then click \"Save names\" when you're happy.</b>"
))
display(output_area)
refresh()

## Step 9 — Diversity Subsampling per Class

**What we're doing:** a cluster covering a long static screen can have dozens-to-hundreds of near-identical member frames — exporting all of them would waste upload bandwidth and blow past the "~20-25 example images per class" guidance already on vid2log's Train page. Instead, greedily pick a spread-out subset in embedding space (farthest-point sampling) so the exported images are genuinely *different-looking* examples of that class, not near-duplicates of each other.

In [ ]:
def farthest_point_sample(indices, embeddings, k):
    indices = list(indices)
    if len(indices) <= k:
        return indices
    chosen = [indices[0]]
    remaining = set(indices[1:])
    while len(chosen) < k and remaining:
        chosen_embs = embeddings[chosen]
        best_idx, best_dist = None, -1.0
        for idx in remaining:
            d = np.linalg.norm(chosen_embs - embeddings[idx], axis=1).min()
            if d > best_dist:
                best_dist, best_idx = d, idx
        chosen.append(best_idx)
        remaining.discard(best_idx)
    return chosen


MAX_IMAGES_PER_CLASS = 25  # matches vid2log's own "~20-25 example images per class" guidance

## Step 10 — Export to Per-Class Folders

Writes one folder per surviving class (after your renames/merges above), each containing its diversity-sampled images — this shape matches vid2log's `ClassDraft { name, files[] }` exactly, so each folder can become one class in the Train tab. A `manifest.csv` summarizing every class is included alongside them, and everything is zipped for download.

In [ ]:
OUTPUT_DIR = "/content/vid2log_classes"
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

manifest = []
for cluster in review_state["clusters"].values():
    class_name = cluster["name"].strip() or f"Class {cluster['id'] + 1}"
    safe_name = "".join(c if c.isalnum() or c in " _-" else "_" for c in class_name).strip()
    safe_name = safe_name or f"class_{cluster['id']}"
    class_dir = os.path.join(OUTPUT_DIR, safe_name)
    os.makedirs(class_dir, exist_ok=True)

    picked = farthest_point_sample(cluster["member_indices"], normed_embeddings, MAX_IMAGES_PER_CLASS)
    for n, idx in enumerate(picked):
        out_path = os.path.join(class_dir, f"frame_{n:03d}.jpg")
        cv2.imwrite(out_path, frames[idx])

    manifest.append({
        "class_name": class_name,
        "folder": safe_name,
        "total_frames_in_video": len(cluster["member_indices"]),
        "images_exported": len(picked),
    })

manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(os.path.join(OUTPUT_DIR, "manifest.csv"), index=False)
manifest_df

In [ ]:
shutil.make_archive("/content/vid2log_classes", "zip", OUTPUT_DIR)
print(f"Zipped {len(manifest)} classes to /content/vid2log_classes.zip")

# Uncomment to download directly in Colab:
# from google.colab import files
# files.download("/content/vid2log_classes.zip")

## Where this fits into vid2log, and what's next

Each exported subfolder here is meant to become exactly one `ClassDraft` on the Train tab (`frontend/app/train/page.tsx`): the folder name becomes the class name, and its images become that class's `files[]` — unzip and drag each folder's contents into a new class on that page to get a real model trained from a single demo video instead of manual per-class uploads.

**Known limitations to keep in mind** (see `Demo_Video_To_Training_Classes_Research.md` for the fuller writeup):
- Highly dynamic screens (animated backgrounds, live counters, moving cursors) can still get over-segmented into more clusters than there really are true classes — that's exactly what the merge step above is for, not a bug to "fix away" with a smarter algorithm alone.
- `MIN_CLUSTER_SIZE` is the one hyperparameter most worth tuning per app/video pacing.

**Reasonable next upgrades, roughly in order of effort:**
1. Swap BLIP for **Moondream2** or **Florence-2** (both free, self-hosted, better at short/precise labels) or the commented Gemini block above (hosted, best label quality, small per-video cost).
2. Fold vid2log's existing OCR extraction into the clustering signal alongside DINOv2 embeddings — two screens with the same layout but different on-screen text are a good hint they're *not* really the same class.
3. Add a **PySceneDetect** pre-filter before Step 2 for longer videos, to avoid embedding long static stretches frame-by-frame.